# Welcome to the GEP Generator

This Jupyter based interface has been designed to support scenario runs for the Global Electrification Platform. 

The interface is built upon a modified version of [OnSSET](http://www.onsset.org/) to provide an easy and quick way to generate electrification investment scenarios compatible with GEP data guidelines and protocols. 

Follow the steps below to generate custom electrification investment outlooks for your country of interest.

#### Start by importing the code 

In [ ]:
from onsset import *
%run funcs.ipynb
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module='matplotlib')

**Note**: In case you encounter an error please refer to the import section of the onsset.py code.

# 1. GIS data selection

First, run the cell below to browse to the directory your input CSV file is located at and select the input file. Sample file shall be located at .\ gep-onsset\test_data. 


In [ ]:
import tkinter as tk
from tkinter import filedialog, messagebox
#from openpyxl import load_workbook
root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)
messagebox.showinfo('OnSSET', 'Open the input file with extracted GIS data')
input_file = filedialog.askopenfilename()

onsseter = SettlementProcessor(input_file)

onsseter.conditioning()

In [ ]:
messagebox.showinfo('OnSSET', 'Open the file with hourly PV data')
pv_path = filedialog.askopenfilename()

messagebox.showinfo('OnSSET', 'Open the file with hourly Wind data')
wind_path = filedialog.askopenfilename()

# 2. Modelling period and target electrification rate

Next, define the modelling period and the electrification rate to be achieved by the end of the analysis. 

In [ ]:
start_year = 2024
intermediate_year = 2027
end_year = 2030

intermediate_electrification_rate_target = 0.70 # E.g. 1 for 100% electrification rate or 0.80 for 80% electrification rate
end_electrification_rate_target = 1 # E.g. 1 for 100% electrification rate or 0.80 for 80% electrification rate 

yearsofanalysis = [intermediate_year, end_year]
eleclimits = {intermediate_year: intermediate_electrification_rate_target, end_year: end_electrification_rate_target}
time_steps = {intermediate_year: intermediate_year-start_year, end_year: end_year-intermediate_year}

# 3. Enter country specific data

In addition to the levers above the user can customize a large number of variables describing the social - economic - technological environment in the selected country. 

**Note!** Most input values shall represent future estimates for the variable, i.e. they describe and **NOT** current values.

### a. Demographics and Social components

In [ ]:
pop_start_year = 33244414       ### Write the population in the base year (e.g. 2020) 
end_year_pop = 38689102         ### Write the expected population in the end year (e.g. 2030) 

urban_ratio_start_year = 0.3486 ### Write the urban population population ratio in the base year (e.g. 2020)
urban_ratio_end_year = 0.3580     ### Write the urban population population ratio in the end year (e.g. 2030)

num_people_per_hh_urban = 4.7     ### Write the number of people per household expected in the end year (e.g. 2030)
num_people_per_hh_rural = 4.5   ### Write the number of people per household expected in the end year (e.g. 2030)

grid_elec_ratio_start_year = 0.505   ### Write the grid electrification rate in the base year (e.g. 2020)
grid_urban_elec_ratio = 0.91      ### Write urban grid electrification rate in the base year (e.g. 2020)
grid_rural_elec_ratio = 0.07         ### Write rural grid electrification rate in the base year (e.g. 2020)

### b. Technology specifications & costs

The cell below contains all the information that is used to calculate the levelized costs for all the technologies, including grid. These default values should be updated to reflect the most accurate values in the country. There are currently 7 potential technologies to include in the model:
* Grid
* PV Hybrid Mini-grid
* Wind Hybrid Mini-grid
* Hydro Mini-grid
* PV Stand-alone systems
* Diesel Stand-alone systems

#### Centralized grid parameters

In [ ]:
grid_generation_cost = 0.10 ### This is the grid cost electricity USD/kWh as expected in the end year of the analysis
grid_power_plants_capital_cost = 2500   ### The cost in USD/kW is for capacity upgrades of the grid
grid_losses = 0.15                     ### The fraction of electricity lost in transmission and distribution (percentage)  

annual_new_grid_connections_limit = 450000 # This is the maximum amount of new households that can be connected to the grid in one year
annual_grid_cap_gen_limit = 9999999     # This is the maximum generation capacity (MW) that can be added to the grid in one year

In [ ]:
# Grid Transmission and distribution costs
hv_line_capacity=110 # kV
hv_line_cost=106000 # USD/km

grid_mv_line_capacity=33 # kV
grid_mv_line_cost = 25000 # USD/kW
grid_mv_line_max_length=70 # km
grid_MV_line_amperage_limit = 275  # Ampere (A)

grid_lv_line_capacity=0.4 #kV
grid_lv_line_max_length=1 # km
grid_lv_line_cost=15000 # USD/km

grid_service_Transf_type=75  # kVA
grid_service_Transf_cost=9000  # $/unit
grid_max_nodes_per_serv_trans=95  # maximum number of nodes served by each service (MV/LV) transformer

hv_mv_transformer_type = 16000 #kVA
hv_mv_transformer_cost = 980000 # USD/unit

#### Off-grid technology parameters

In [ ]:
min_mg_size = 100             # Minimum number of connections (people) for mini-grids to be considered

# Decide whether mini-grids should be allowed to be interconnected to the grid in a later time-step.
mg_interconnection = 0         # 0 = NO, 1 = YES

In [ ]:
diesel_price = 1.43                    ### This is the diesel price in USD/liter as expected in the end year of the analysis

sa_diesel_cost = {'diesel_price': diesel_price,
                  'efficiency': 0.28,
                  'diesel_truck_consumption': 14,
                  'diesel_truck_volume': 300}

mg_diesel_cost = {'diesel_price': diesel_price,
                  'efficiency': 0.33,
                  'diesel_truck_consumption': 14,
                  'diesel_truck_volume': 15000}

In [ ]:
diesel_techs = 0                                  ### 0 = Stand-alone diesel NOT included, 1 = stand-alone diesel included 

sa_diesel_capital_cost = {float("inf"): 938}      ### Stand-alone Diesel capital cost (USD/kW) as expected in the years of the analysis
mg_hydro_capital_cost = {float("inf"): 15000}      ### Mini-grid Hydro capital cost (USD/kW) as expected in the years of the analysis

In [ ]:
# PV and Wind hybrid mini-grid costs
pv_cost = 1400                     # PV panel costs including BoS (PV inverter, charge controller) (USD/kW)
battery_cost = 550                 # battery capital cost, USD/kWh of storage capacity                    
inverter_cost  = 598               # Battery inverter, USD/kW
diesel_gen_cost = 500             # diesel generator capital cost, USD/kW rated power

wind_cost = 14000                   # Wind turbine capital cost, USD/kW peak power

inverter_life=20    # Battery inverter expected lifetime in mini-grid, years
diesel_life=20      # diesel generator expected lifetime in mini-grid, years
pv_life=25          # PV panel expected lifetime in mini-grid, years
        
lpsp_max=0.10         # maximum loss of load allowed over the year, in share of kWh (e.g. 0.1 means that the mini-grid will be able to meet at least 90% of the demand over the year)
max_diesel = 0.5     # Maximum share of generation that can come from diesel generators (0-1). Set to 0 for fully renewable mini-grids

In [ ]:
# Mini-grid distribution network spcification

mg_mv_line_capacity=33 # kV
mg_mv_line_cost = 25000 # USD/kW
mg_MV_line_amperage_limit = 275  # Ampere (A)

mg_lv_line_capacity=0.4 #kV
mg_lv_line_max_length=1 # km
mg_lv_line_cost=12000 # USD/km

mg_service_Transf_type=75  # kVA
mg_service_Transf_cost=9000  # $/unit
mg_max_nodes_per_serv_trans=95  # maximum number of nodes served by each service (MV/LV) transformer

In [ ]:
sa_pv_capital_cost_1 = 11600          ### Stand-alone PV capital cost (USD/kW) for household systems under 20 W
sa_pv_capital_cost_2 = 7500          ### Stand-alone PV capital cost (USD/kW) for household systems between 21-50 W
sa_pv_capital_cost_3 = 7500          ### Stand-alone PV capital cost (USD/kW) for household systems between 51-100 W
sa_pv_capital_cost_4 = 8000           ### Stand-alone PV capital cost (USD/kW) for household systems between 101-1000 W
sa_pv_capital_cost_5 = 8000           ### Stand-alone PV capital cost (USD/kW) for household systems over 1 kW

#### Discount rates

In [ ]:
grid_discount_rate = 0.12 # E.g. 0.08 means a discount rate of 8%
mini_grid_discount_rate = 0.12
standalone_discount_rate = 0.12 

#### Additional technology specifications

In [ ]:
# Centralized grid costs
grid_calc = Technology(om_of_td_lines=0.02,
                       distribution_losses=grid_losses,
                       connection_cost_per_hh=60,
                       base_to_peak_load_ratio=0.8,
                       capacity_factor=1,
                       tech_life=30,
                       grid_penalty_ratio=1,
                       grid_capacity_investment=grid_power_plants_capital_cost,
                       grid_price=grid_generation_cost,
                       discount_rate=grid_discount_rate,
                       mv_line_type=grid_mv_line_capacity,
                       mv_line_amperage_limit=grid_MV_line_amperage_limit, 
                       mv_line_cost=grid_mv_line_cost, 
                       lv_line_type=grid_lv_line_capacity, 
                       lv_line_cost=grid_lv_line_cost, 
                       lv_line_max_length=grid_lv_line_max_length, 
                       service_transf_type=grid_service_Transf_type, 
                       service_transf_cost=grid_service_Transf_cost,
                       max_nodes_per_serv_trans=grid_max_nodes_per_serv_trans)

mg_pv_hybrid_calc = Technology(om_of_td_lines=0.02,
                               distribution_losses=0.05,
                               connection_cost_per_hh=100,
                               capacity_factor=0.5,
                               tech_life=25,
                               mini_grid=True,
                               hybrid=True,
                               discount_rate=mini_grid_discount_rate,
                               mv_line_type=mg_mv_line_capacity,
                               mv_line_amperage_limit=mg_MV_line_amperage_limit, 
                               mv_line_cost=mg_mv_line_cost, 
                               lv_line_type=mg_lv_line_capacity, 
                               lv_line_cost=mg_lv_line_cost, 
                               lv_line_max_length=mg_lv_line_max_length, 
                               service_transf_type=mg_service_Transf_type, 
                               service_transf_cost=mg_service_Transf_cost,
                               max_nodes_per_serv_trans=mg_max_nodes_per_serv_trans)

mg_wind_hybrid_calc = Technology(om_of_td_lines=0.02,
                                 distribution_losses=0.05,
                                 connection_cost_per_hh=100,
                                 capacity_factor=0.5,
                                 tech_life=20,
                                 mini_grid=True,
                                 hybrid=True,
                                 discount_rate=mini_grid_discount_rate,
                                 mv_line_type=mg_mv_line_capacity,
                                 mv_line_amperage_limit=mg_MV_line_amperage_limit, 
                                 mv_line_cost=mg_mv_line_cost, 
                                 lv_line_type=mg_lv_line_capacity, 
                                 lv_line_cost=mg_lv_line_cost, 
                                 lv_line_max_length=mg_lv_line_max_length, 
                                 service_transf_type=mg_service_Transf_type, 
                                 service_transf_cost=mg_service_Transf_cost,
                                 max_nodes_per_serv_trans=mg_max_nodes_per_serv_trans)

# Mini-grid hydro costs
mg_hydro_calc = Technology(om_of_td_lines=0.02,
                            distribution_losses=0.05,
                            connection_cost_per_hh=100,
                            base_to_peak_load_ratio=0.85,
                            capacity_factor=0.5,
                            tech_life=30,
                            capital_cost=mg_hydro_capital_cost,
                            om_costs=0.02,
                            discount_rate=mini_grid_discount_rate,
                            mv_line_type=mg_mv_line_capacity,
                            mv_line_amperage_limit=mg_MV_line_amperage_limit, 
                            mv_line_cost=mg_mv_line_cost, 
                            lv_line_type=mg_lv_line_capacity, 
                            lv_line_cost=mg_lv_line_cost, 
                            lv_line_max_length=mg_lv_line_max_length, 
                            service_transf_type=mg_service_Transf_type, 
                            service_transf_cost=mg_service_Transf_cost,
                            max_nodes_per_serv_trans=mg_max_nodes_per_serv_trans
                            )

# Stand-alone PV costs
sa_pv_calc = Technology(base_to_peak_load_ratio=0.9,
                        tech_life=5,
                        om_costs=0.05,
                        discount_rate=standalone_discount_rate,
                        capital_cost={0.020: sa_pv_capital_cost_1, 
                                      0.050: sa_pv_capital_cost_2, 
                                      0.100: sa_pv_capital_cost_3, 
                                      1: sa_pv_capital_cost_4, 
                                      float("inf"): sa_pv_capital_cost_5},
                        standalone=True
                        )

# Stand-alone diesel costs
sa_diesel_calc = Technology(base_to_peak_load_ratio=0.9,
                            capacity_factor=0.7,
                            tech_life=10,
                            om_costs=0.1,
                            capital_cost=sa_diesel_capital_cost,
                            diesel_price=diesel_price,
                            standalone=True,
                            efficiency=0.28,
                            diesel_truck_consumption=14,
                            diesel_truck_volume=300)

### c. Electricity demand target

For the second lever, enter the target tier (level of electricity access) for urban and rural households respectively. This can take a value between "1" (lowest level of electricity access) and "5" (highest level of electricity access) as in ESMAPs Multi-Tier Framework for Measuring Electricity Access (found <a href="https://www.esmap.org/node/55526" target="_blank">here</a>). Alternatively, enter "6" to use a distribution of the tiers across the country based on poverty levels and GDP according to the methodology found <a href="https://www.mdpi.com/1996-1073/12/7/1395" target="_blank">here</a>.   

*On the GEP Explorer, the following three electricity demand targets are used:*

*Top down demand target - Low: In this case, all urban clusters are tergeted to reach the current average consumption level of electrified households, and rural settlements are assigned Tier 1.* 

*Top down demand target - High: In this case, all urban clusters are tergeted to reach one Tier higher than the current average consumption level of electrified households, and rural settlements are assigned Tier 3.*

*Bottom up demand target (Poverty - GDP): In this case each settlement is assigned a demand target based on poverty and GDP levels as described in the methodology above. Choose "6" for both the urban_target_tier and rural_target_tier to use this option* 

In [ ]:
# Define the annual household electricity targets to choose from
tier_1 = 73  # 38.7 refers to kWh/household/year. 
tier_2 = 350
tier_3 = 620
tier_4 = 2117
tier_5 = 3000

In [ ]:
small_rural_cutoff = 100  # Any rural settlement with fewer population is considered a "small rural" settlement, rural settlements with larger population is considered "large rural" 

urban_target_tier = 3               # Target demand Tier for Urban settlements
rural_target_tier_large = 2         # Target demand Tier for large rural settlements
rural_target_tier_small = 1         # Target demand Tier for small rural settlements

In [ ]:
productive_demand = 1                     # 1 if productive demand is defined and should be included, else 0

### d. Rollout plan
This lever reflects the electrification approach to be examined. On the GEP Explorer, there are currently two options in use:

**Nationwide Least Cost approach:** This options aims to achieve the electrification rate targets set for the intermediate and end year. For the years where the target is set below 100%, the algorithm prioritizes grid densification first (ramp up in already electrified clusters) then selection based on lowest invetsment cost per capita to choose which clusters to be electrified.

**Forced grid approach:** Forced grid under a defined buffer zone (auto_intensification set equal to X km) & least cost approach outside of the buffer zone.

In [ ]:
auto_intensification = 30        # Buffer distance (km) for automatic intensification

max_grid_extension_cost = 2000  # Maximum cost per household (USD/household) for grid intensification 

# 4. Preparation for scenario runs

The following steps adds some additional useful information to be used in the further calculations. 

The land cover dict describes how suitable different land cover types are for grid extension. 5 is best, 1 is worst.
The current values are based on the GLC2000 land cover data (https://forobs.jrc.ec.europa.eu/glc2000). If you are using a different land cover dataset, you may need to update accordingly.

In [ ]:
land_cover_dict = {
    0: 0,
    1: 3, # Tree Cover, broadleaved, evergreen
    2: 3, # Tree Cover, broadleaved, deciduous, closed 
    3: 3, #Tree Cover, broadleaved, deciduous, open
    4: 3, # Tree Cover, needle-leaved, evergreen
    5: 3, # Tree Cover, needle-leaved, deciduous
    6: 3, # Tree Cover, mixed leaf type
    7: 1, # Tree Cover, regularly flooded, fresh  water 
    8: 1, # Tree Cover, regularly flooded, saline water
    9: 3, # Mosaic: Tree cover / Other natural vegetation 
    10: 3, # Tree Cover, burnt
    11: 4, # Shrub Cover, closed-open, evergreen
    12: 4, # Shrub Cover, closed-open, deciduous 
    13: 5, # Herbaceous Cover, closed-open 
    14: 5, # Sparse Herbaceous or sparse Shrub Cover
    15: 1, # Regularly flooded Shrub and/or Herbaceous Cover
    16: 4, # Cultivated and managed areas
    17: 3, # Mosaic: Cropland / Tree Cover / Other natural vegetation
    18: 4, # Mosaic: Cropland / Shrub or Grass Cover 
    19: 5, # Bare Areas
    20: 1, # Water Bodies 
    21: 2, # Snow and Ice 
    22: 3 # Artificial surfaces and associated areas
}

#onsseter.condition_df()
onsseter.df['GridPenalty'] = onsseter.grid_penalties(onsseter.df, land_cover_dict)
onsseter.df['WindCF'] = onsseter.calc_wind_cfs()

# 5. Start a scenario run, which calculate and compare technology costs for every settlement in the country

Based on the previous calculation this piece of code identifies the LCoE that every off-grid technology can provide, for each single populated settlement of the selected country. The cell then takes all the currently grid-connected points in the country, and looks at the points within a certain distance from them, to see if it is more economical to connect them to the grid, or to use one of the off-grid technologies calculated above. Once more points are connected to the grid, the process is repeated, so that new points close to those points might also be connected. This is repeated until there are no new points to connect to the grid.

In [ ]:
onsseter.df.PerCapitaDemand = 0

import time

onsseter.project_pop_and_urban(end_year_pop, urban_ratio_end_year, start_year, yearsofanalysis)

prioritization = 5

onsseter.prepare_wtf_tier_columns(tier_1, tier_2, tier_3, tier_4, tier_5)

onsseter.current_mv_line_dist()

try:
    onsseter.df.reset_index(inplace=True)
except ValueError:
    pass

Technology.set_default_values(base_year=start_year, start_year=start_year, end_year=end_year,
                             hv_line_type=hv_line_capacity, hv_line_cost=hv_line_cost, hv_mv_sub_station_cost=hv_mv_transformer_cost,
                             hv_mv_substation_type=hv_mv_transformer_type)

for year in yearsofanalysis:
        
    eleclimit = eleclimits[year]
    time_step = time_steps[year]
    
    grid_connect_limit = time_step * annual_new_grid_connections_limit
    grid_cap_gen_limit = time_step * annual_grid_cap_gen_limit * 1000
        
    onsseter.set_scenario_variables(year, time_step,
                                    start_year, urban_target_tier, 
                                    rural_target_tier_large, rural_target_tier_small, 
                                    productive_demand, small_rural_cutoff)

    try:
        onsseter.diesel_cost_columns(sa_diesel_cost, mg_diesel_cost, year)
    except ValueError:
        pass
    
    print('CalcPVHybrid ', time.ctime())
    
    mg_pv_hybrid_investment, mg_pv_hybrid_capacity = onsseter.calculate_pv_hybrids_lcoe(year, 
                                                                                        year - time_step, 
                                                                                        end_year, 
                                                                                        time_step, 
                                                                                        mg_pv_hybrid_calc,
                                                                                        1, 
                                                                                        pv_path,
                                                                                        max_diesel=max_diesel, 
                                                                                        battery_cost=battery_cost, 
                                                                                        pv_cost=pv_cost, 
                                                                                        inverter_cost=inverter_cost,
                                                                                        diesel_cost=diesel_gen_cost,
                                                                                        discount_rate=mini_grid_discount_rate,
                                                                                        lpsp_max=lpsp_max)
    onsseter.df['PVHybridInvestment{}'.format(year)] = mg_pv_hybrid_investment
    
    print('CalcWindHybrid ', time.ctime())

    mg_wind_hybrid_investment, mg_wind_hybrid_capacity = onsseter.calculate_wind_hybrids_lcoe(year,
                                                                                              year - time_step,
                                                                                              end_year,
                                                                                              time_step,
                                                                                              mg_wind_hybrid_calc,
                                                                                              wind_path,
                                                                                              max_diesel=max_diesel, 
                                                                                              battery_cost=battery_cost, 
                                                                                              wind_cost=wind_cost, 
                                                                                              inverter_cost=inverter_cost,
                                                                                              diesel_cost=diesel_gen_cost,
                                                                                              discount_rate=mini_grid_discount_rate)

    sa_diesel_investment, sa_pv_investment, mg_hydro_investment = onsseter.calculate_off_grid_lcoes(mg_hydro_calc,  
                                                                                                    sa_pv_calc, 
                                                                                                    sa_diesel_calc, 
                                                                                                    year, 
                                                                                                    end_year, 
                                                                                                    time_step,
                                                                                                    diesel_techs,
                                                                                                    min_mg_size=min_mg_size)
    
    print('Grid extension ', time.ctime())
    
    grid_investment, grid_cap_gen_limit, grid_connect_limit = onsseter.pre_electrification(grid_generation_cost, 
                                                                                           year, 
                                                                                           time_step, 
                                                                                           end_year, 
                                                                                           grid_calc, 
                                                                                           grid_cap_gen_limit,
                                                                                           grid_connect_limit)

    onsseter.df[SET_LCOE_GRID + "{}".format(year)], onsseter.df[SET_MIN_GRID_DIST + "{}".format(year)], \
    onsseter.df[SET_ELEC_ORDER + "{}".format(year)], onsseter.df[SET_MV_CONNECT_DIST], grid_investment, \
    onsseter.df['GridLCOEActual{}'.format(year)]= \
    onsseter.elec_extension(grid_calc,
                            grid_mv_line_max_length,
                            year,
                            start_year,
                            end_year,
                            time_step,
                            grid_cap_gen_limit,
                            grid_connect_limit,
                            auto_intensification=auto_intensification,
                            prioritization=prioritization,
                            new_investment=grid_investment,
                            threshold=max_grid_extension_cost)

    onsseter.results_columns(year, time_step, prioritization, auto_intensification, mg_interconnection)

    onsseter.calculate_investments(sa_diesel_investment, sa_pv_investment, mg_hydro_investment, 
                                   mg_pv_hybrid_investment, mg_wind_hybrid_investment, grid_investment, year)
    
    onsseter.calculate_new_capacity(mg_pv_hybrid_capacity, mg_wind_hybrid_capacity, mg_hydro_calc, sa_pv_calc, 
                                    sa_diesel_calc, grid_calc, year)

    grid_connect_limit = time_step * annual_new_grid_connections_limit
    grid_cap_gen_limit = time_step * annual_grid_cap_gen_limit * 1000
    
    if year == yearsofanalysis[-1]:
        final_step = True
    else:
        final_step = False
    
    onsseter.check_grid_limitations(grid_connect_limit, grid_cap_gen_limit, year, time_step, final_step)
    
    onsseter.apply_limitations(eleclimit, year, time_step, prioritization, auto_intensification, SET_POP)

    onsseter.tech_code_update_jn(year, time_step)

onsseter.df = finalize_results(onsseter.df, yearsofanalysis)

# 6. Results, Summaries and Visualization
With all the calculations and grid-extensions complete, this block gets the final results on which technology was chosen for each point, how much capacity needs to be installed and what it will cost. Then the summaries, plots and maps are generated.

In [ ]:
summary_table, columns = calc_summary_table(onsseter.df, yearsofanalysis, start_year)

additional_info_index = ['', 'Existing grid settlements', 'Grid-extension settlements', 'New PV mini-grids', 'New Hydro mini-grids', 'New Wind mini-grids', 'SHS settlements']

df = pd.DataFrame(index=additional_info_index, columns=columns)
df.loc['Existing grid settlements'][0] = len(onsseter.df.loc[(onsseter.df['FinalElecCode' + "{}".format(start_year)] == 1)])
df.loc['Grid-extension settlements'][0] = len(onsseter.df.loc[(onsseter.df['FinalElecCode' + "{}".format(end_year)] == 10) & (onsseter.df['FinalElecCode' + "{}".format(start_year)] != 10)])
df.loc['New PV mini-grids'][0] = len(onsseter.df.loc[(onsseter.df['FinalElecCode' + "{}".format(end_year)] == 8) & (onsseter.df['FinalElecCode' + "{}".format(start_year)] != 8)])
df.loc['New Hydro mini-grids'][0] = len(onsseter.df.loc[(onsseter.df['FinalElecCode' + "{}".format(end_year)] == 7) & (onsseter.df['FinalElecCode' + "{}".format(start_year)] != 7)])
df.loc['New Wind mini-grids'][0] = len(onsseter.df.loc[(onsseter.df['FinalElecCode' + "{}".format(end_year)] == 9) & (onsseter.df['FinalElecCode' + "{}".format(start_year)] != 9)])
df.loc['SHS settlements'][0] = len(onsseter.df.loc[(onsseter.df['FinalElecCode' + "{}".format(end_year)] == 3) & (onsseter.df['FinalElecCode' + "{}".format(start_year)] != 3)])
df2 = pd.concat([summary_table, df], ignore_index=False)

summary_table

In [ ]:
bar_plot(summary_table, columns)

In [ ]:
map_plot(onsseter.df, start_year)

## 7. Exporting results

This code generates three csv files:
 - one containing all the results for the scenario created
 - one containing the summary for the scenario created
 - one containing some if the key input variables of the scenario

Before we proceed, please write the scenario_name in the first cell below. then move on to the next cell and run it to browse to the directory where you want to save your results. Sample file shall be located at .\ gep-onsset\sample_output. 

**Note that if you do not change the scenario name, the previous output files will be overwritten**

In [ ]:
scenario_name = "Scenario_X"

In [ ]:
list1 = [('Start_year',start_year,'','',''), 
         ('End_year',end_year,'','',''),
         ('End year electrification rate target',end_electrification_rate_target,'','',''),
         ('Intermediate target year', intermediate_year,'','',''),
         ('Intermediate electrification rate target', intermediate_electrification_rate_target,'','',''),
         ('Urban target tier', urban_target_tier, '', '', ''),
         ('Rural target tier (large)', rural_target_tier_large, '', '', ''),
         ('Rural target tier (small)', rural_target_tier_small, '', '', ''),
         ('Rural threshold', small_rural_cutoff, '', '', "Any rural settlement with fewer population is considered a small rural settlement, rural settlements with larger population is considered large rural"),
         ('Auto intensification distance', auto_intensification, '', '', 'Buffer distance (km) for automatic intensification if choosing prioritization 1'),
         ('Grid discount_rate', grid_discount_rate,'','',''),
         ('Mini-grid discount_rate', mini_grid_discount_rate,'','',''),
         ('SHS discount_rate', standalone_discount_rate,'','',''),
         ('pop_start_year',pop_start_year,'','','the population in the base year (e.g. 2018)'),
         ('pop_end_year',end_year_pop,'','','the projected population in the end year (e.g. 2030)'),
         ('urban_ratio_start_year',urban_ratio_start_year,'','','the urban population population ratio in the base year (e.g. 2018)'),
         ('urban_ratio_end_year',urban_ratio_end_year,'','','the urban population population ratio in the end year (e.g. 2030)'),
         ('num_people_per_hh_urban',num_people_per_hh_urban,'','','the number of people per household expected in the end year (e.g. 2030)'),
         ('num_people_per_hh_rural',num_people_per_hh_rural,'','','the number of people per household expected in the end year (e.g. 2030)'),
         ('elec_ratio_start_year',grid_elec_ratio_start_year,'','','the electrification rate in the base year (e.g. 2018)'),
         ('urban_elec_ratio',grid_urban_elec_ratio,'','','urban electrification rate in the base year (e.g. 2018)'),
         ('rural_elec_ratio',grid_rural_elec_ratio,'','','rural electrification rate in the base year (e.g. 2018)'),
         ('grid_generation_cost', grid_generation_cost,'','','This is the grid cost electricity USD/kWh as expected in the end year of the analysis'),
         ('grid_power_plants_capital_cost', grid_power_plants_capital_cost,'','','The cost in USD/kW to for capacity upgrades of the grid-connected power plants'),
         ('grid_losses',grid_losses,'','','The fraction of electricity lost in transmission and distribution (percentage)'),
         ('HV_line_cost', hv_line_cost, '', '', 'USD/km'),
         ('hv_line_capacity', hv_line_capacity, '', '', 'kV'),
         ('grid_mv_line_capacity', grid_mv_line_capacity, '', '', 'Capacity of MV lines for grid (kV)'),
         ('grid_mv_line_cost', grid_mv_line_cost, '', '', 'Cost of MV lines for grid (USD/km)'),
         ('grid_mv_line_max_length', grid_mv_line_max_length, '', '', 'Maximum length of MV lines (km)'),
         ('grid_MV_line_amperage_limit', grid_MV_line_amperage_limit, '', '', '(A)'),
         ('grid_lv_line_capacity', grid_lv_line_capacity, '', '', 'Capacity of LV lines for grid (kV)'),
         ('grid_lv_line_cost', grid_lv_line_cost, '', '', 'Cost of LV lines for grid (USD/km)'),
         ('grid_lv_line_max_length', grid_lv_line_max_length, '', '', 'Maximum length of LV lines (km)'),
         ('grid_service_Transf_type', grid_service_Transf_type, '', '', 'Caapcity of service transformer (MV/LV) for grid (kVA)'),
         ('grid_service_Transf_cost', grid_service_Transf_cost, '', '', 'Cost per service transformer for grid (USD)'),
         ('grid_max_nodes_per_serv_trans', grid_max_nodes_per_serv_trans, '', '', 'Maximum number of customers served by each service (MV/LV) transformer'),
         ('hv_mv_transformer_type', hv_mv_transformer_type, '', '', 'Capacity of HV/MV transformer (kVA)'),
         ('hv_mv_transformer_cost', hv_mv_transformer_cost, '', '', 'Cost per HV/MV transformer (USD)'),
         ('annual_new_grid_connections_limit', annual_new_grid_connections_limit,'','','This is the maximum amount of new households that can be connected to the grid in one year (thousands)'),
         ('annual_grid_capacity_limit_end',annual_grid_cap_gen_limit,'','','This is the maximum generation capacity that can be added to the grid in one year (MW)'),
         ('diesel_price',diesel_price,'','','This is the diesel price in USD/liter as expected in the end year of the analysis'),
         ('Minimum mini-grid size (population)', min_mg_size , '', '', ''),
         ('Mini-grid interconnection', mg_interconnection, '', '', 'Whether mini-grids should be allowed to be interconnected to the grid in a later time-step. 0 = NO, 1 = YES'),
         ('Mini-grid pv_cost', pv_cost, '', '', 'PV panel costs including BoS (PV inverter, charge controller) (USD/kW)'),
         ('Mini-grid battery cost', battery_cost, '', '', 'Battery capital cost, USD/kWh of storage capacity'),
         ('Mini-grid battery inverter', inverter_cost, '', '', 'Battery inverter, USD/kW'),
         ('Mini-grid diesel_generator_cost', diesel_gen_cost, '', '', 'diesel generator capital cost (USD/kVA)'),
         ('Mini-grid wind_cost', wind_cost, '', '', 'Wind turbine capital cost, USD/kW peak power'),
         ('Mini-grid max_diesel share', max_diesel, '', '', 'Maximum share of generation that can come from diesel generators (0-1).'),
         ('Mini-grid LPSP max', lpsp_max, '', '', ''),
         ('Mini-grid inverter_life', inverter_life, '', '', 'Battery inverter expected lifetime in mini-grid, years'),
         ('Mini_grid diesel_life', diesel_life, '', '', 'diesel generator expected lifetime in mini-grid, years'),
         ('Mini-grid pv_life', pv_life, '', '', 'PV panel expected lifetime in mini-grid, years'),
         ('Mini-grid mv_line_capacity', mg_mv_line_capacity, '', '', 'Capacity of MV lines for Mini-grid (kV)'),
         ('Mini-grid mv_line_cost', mg_mv_line_cost, '', '', 'Cost of MV lines for Mini-grid (USD/km)'),
         ('Mini-grid MV_line_amperage_limit', mg_MV_line_amperage_limit, '', '', '(A)'),
         ('Mini-grid LV_line_capacity', mg_lv_line_capacity, '', '', 'Capacity of LV lines for Mini-grid (kV)'),
         ('Mini-grid LV_line_cost', mg_lv_line_cost, '', '', 'Cost of LV lines for Mini-grid (USD/km)'),
         ('Mini-grid LV_line_max_length', mg_lv_line_max_length, '', '', 'Maximum length of LV lines (km) for Mini-grid'),
         ('Mini-grid service_Transf_type', mg_service_Transf_type, '', '', 'Caapcity of service transformer (MV/LV) for Mini-grid (kVA)'),
         ('Mini-grid service_Transf_cost', mg_service_Transf_cost, '', '', 'Cost per service transformer for Mini-grid (USD)'),
         ('Mini-grid max_nodes_per_serv_trans', mg_max_nodes_per_serv_trans, '', '', 'Maximum number of customers served by each mni-grid service (MV/LV) transformer'),
         ('mg_hydro_capital_cost',mg_hydro_capital_cost,'','','Mini-grid Hydro capital cost (USD/kW) as expected in the years of the analysis'),
         ('sa_pv_capital_cost_1',sa_pv_capital_cost_1,'','','Stand-alone PV capital cost (USD/kW) for household systems under 20 W'),
         ('sa_pv_capital_cost_2',sa_pv_capital_cost_2,'','','Stand-alone PV capital cost (USD/kW) for household systems between 21-50 W'),
         ('sa_pv_capital_cost_3',sa_pv_capital_cost_3,'','','Stand-alone PV capital cost (USD/kW) for household systems between 51-100 W'),
         ('sa_pv_capital_cost_4',sa_pv_capital_cost_4,'','','Stand-alone PV capital cost (USD/kW) for household systems between 101-200 W'),
         ('sa_pv_capital_cost_5',sa_pv_capital_cost_5,'','','Stand-alone PV capital cost (USD/kW) for household systems over 200 W'),
         ('SHS technology lifetime', sa_pv_calc.tech_life, '', '', 'Stand-alone PV (SHS) expected technology lifetime (years)')
         ]
labels = ['Variable','Value', 'Source', 'Comments', 'Description']
df_variables = pd.DataFrame.from_records(list1, columns=labels)

In [ ]:
messagebox.showinfo('OnSSET', 'Browse to the folder where you want to save the outputs')

output_dir = filedialog.askdirectory()
output_dir_variables = os.path.join(output_dir, '{}_Variables.csv'.format(scenario_name))
output_dir_results = os.path.join(output_dir, '{}_Results.csv'.format(scenario_name))
output_dir_summaries = os.path.join(output_dir, '{}_Summaries.csv'.format(scenario_name))

In [ ]:
# Returning the result as a csv file
onsseter.df.to_csv(output_dir_results, float_format="%.4f", index=False)

# Returning the summary as a csv file
df2.to_csv(output_dir_summaries, index=True)

# Returning the input variables as a csv file
df_variables.to_csv(output_dir_variables, index=False)

In [ ]:
prov = onsseter.df.Province.unique()

In [ ]:
for p in prov:
    df_prov = onsseter.df.loc[onsseter.df.Province == p]
    df_prov.to_csv(os.path.join(output_dir, '{}_Results.csv'.format(p)), float_format="%.4f", index=False)